In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

In [2]:
# Simulation parameters

ENGY=[50, 200] # Energy range in Watt-hrs
LONG=[0, 1000] # Longitude 10 km
LAT= [0, 1000] # Latitude 10 km
ALT=[5, 20] # Altitude 0.5 km ie 500 m

#Inital reputation score is based on below resources
clock_speed_range=[5, 26] # Clock speed is from 5 to 15 MHz
memory_range=[2, 17] #RAM size is 2 to 16 GB
power_consumption_range = [100, 150]

# Total number of trails
TRIALS = 100

# Communication parameters
FREQUENCY = 2.4e9  # Frequency in Hz (2.4 GHz for WiFi)
BANDWIDTH = 20e6  # Bandwidth in Hz (20 MHz)
TRANSMISSION_POWER = 1  # Transmission power in Watts
NOISE_POWER = 1e-9  # Noise power in Watts (assumed)
SPEED_OF_LIGHT = 3e8  # Speed of light in meters per second
r=0.7 # Reputation weight
d=0.3 # Data Rate weight

def calculate_distance(uav1, uav2):
  """Calculates the Euclidean distance between two UAVs."""
  x1, y1, z1 = uav1.longitude, uav1.latitude, uav1.altitude
  x2, y2, z2 = uav2.longitude, uav2.latitude, uav2.altitude
  return np.sqrt((x1 - x2)**2 + (y1 - y2)**2 + (z1 - z2)**2)

def calculate_data_rate(distance):
  """Calculates the data rate between two UAVs given their distance."""
  path_loss = (SPEED_OF_LIGHT / (4 * np.pi * FREQUENCY * distance))**2
  snr = TRANSMISSION_POWER * path_loss / NOISE_POWER
  data_rate = BANDWIDTH * np.log2(1 + snr)
  return data_rate

# Function to update positions of UAVs
def update_uav_positions(uavs, long_range, lat_range, alt_range):
    for uav in uavs:
        uav.longitude = np.random.uniform(long_range[0], long_range[1])
        uav.latitude = np.random.uniform(lat_range[0], lat_range[1])
        uav.altitude = np.random.uniform(alt_range[0], alt_range[1])

In [3]:
# prompt: generate a class named uav, with the components of residual energy, memory, clock speed, logitude, latitude and altitude

class UAV:
  def __init__(self, residual_energy, memory, clock_speed, longitude, latitude, altitude, avg_power_consumption):
    self.residual_energy = residual_energy
    self.memory = memory
    self.clock_speed = clock_speed
    self.longitude = longitude
    self.latitude = latitude
    self.altitude = altitude
    self.avg_power_consumption = avg_power_consumption
    self.minimum_energy = residual_energy/10
    self.select_count = 0

class Task:
  def __init__(self, req_clock_speed, req_memory, max_latency, data_volume, instructions_per_unit_data):
    self.req_clock_speed = req_clock_speed
    self.req_memory = req_memory
    self.max_latency = max_latency
    self.data_volume = data_volume
    self.req_data_rate = data_volume / max_latency
    self.instructions_per_unit_data = instructions_per_unit_data
    self.total_instructions = instructions_per_unit_data*data_volume


In [4]:
# prompt: declare a variable num_uavs and create those many uavs with random parameters (i have LONG as the array of min and max ranges, LAT as the min and max ranges, and same the same thing for memory, clock_speed, and ALT for altitude)

num_uavs = 20
uavs = []
for _ in range(num_uavs):
  residual_energy = np.random.uniform(ENGY[0], ENGY[1])
  memory = np.random.randint(memory_range[0], memory_range[1])
  clock_speed = np.random.randint(clock_speed_range[0], clock_speed_range[1])
  longitude = np.random.uniform(LONG[0], LONG[1])
  latitude = np.random.uniform(LAT[0], LAT[1])
  altitude = np.random.uniform(ALT[0], ALT[1])
  avg_power_consumption = np.random.uniform(power_consumption_range[0], power_consumption_range[1])
  uavs.append(UAV(residual_energy, memory, clock_speed, longitude, latitude, altitude, avg_power_consumption))


In [5]:
# prompt: calculate initial reputation of uav as clock_speed*memory

# Calculate initial reputation of UAVs
uav_reputations = {}
for i, uav in enumerate(uavs):
  uav_reputations[i] = uav.clock_speed * uav.memory

# Print initial reputations
print("Initial UAV Reputations:")
for uav_idx, reputation in uav_reputations.items():
  print(f"UAV {uav_idx}: {reputation}")


Initial UAV Reputations:
UAV 0: 168
UAV 1: 195
UAV 2: 52
UAV 3: 90
UAV 4: 132
UAV 5: 92
UAV 6: 168
UAV 7: 119
UAV 8: 207
UAV 9: 220
UAV 10: 168
UAV 11: 162
UAV 12: 100
UAV 13: 165
UAV 14: 100
UAV 15: 28
UAV 16: 208
UAV 17: 33
UAV 18: 144
UAV 19: 10


In [6]:
# prompt: calculate the data rate between every 2 uavs, i have only their positions. declare all the other variables required

# Calculate data rates between all pairs of UAVs
data_rates = np.zeros((num_uavs, num_uavs))
for i in range(num_uavs):
  for j in range(i + 1, num_uavs):
    distance = calculate_distance(uavs[i], uavs[j])
    data_rate = calculate_data_rate(distance)
    data_rates[i, j] = data_rate
    data_rates[j, i] = data_rate  # Symmetric matrix

# Print the data rate matrix
print("Data Rate Matrix (in bits per second):\n", data_rates)


Data Rate Matrix (in bits per second):
 [[0.00000000e+00 4.20542448e+07 4.61830044e+07 1.73604543e+07
  1.02065694e+08 1.89224119e+07 4.12494414e+07 7.92444334e+06
  2.39067377e+07 6.85802630e+06 1.78100621e+07 4.38757566e+06
  5.44986418e+06 2.11597660e+07 4.87929301e+06 4.39741912e+06
  6.46090997e+06 1.96165248e+07 2.59668166e+07 8.14487543e+06]
 [4.20542448e+07 0.00000000e+00 3.68261978e+07 2.19733446e+07
  5.97583075e+07 1.57328974e+07 5.07717268e+07 7.31371381e+06
  2.33512935e+07 4.61871282e+06 9.71583448e+06 3.41133063e+06
  4.34155284e+06 4.08844369e+07 3.71796239e+06 3.35844623e+06
  5.26796139e+06 9.93632531e+06 6.44183371e+07 4.99054116e+06]
 [4.61830044e+07 3.68261978e+07 0.00000000e+00 3.66303088e+07
  5.05921992e+07 3.97832396e+07 8.70020408e+07 1.32652673e+07
  5.99988995e+07 7.46459237e+06 1.06448379e+07 5.40251911e+06
  7.32155091e+06 3.37641633e+07 6.00012860e+06 5.27443912e+06
  9.31609058e+06 1.52842770e+07 3.35766763e+07 6.52629013e+06]
 [1.73604543e+07 2.19733446

In [7]:
# prompt: take a variable num_taks and generate the tasks with the random inputs

# Ranges of task components
req_clock_speed_range = [3, 5]
req_memory_range = [1, 4]
max_latency_range = [0.001, 0.01]
data_volume_range = [10, 100]
instructions_per_unit_data_range = [1e6, 2e6]

num_tasks = 5
tasks = []
for _ in range(num_tasks):
  req_clock_speed = np.random.randint(req_clock_speed_range[0], req_clock_speed_range[1])
  req_memory = np.random.randint(req_memory_range[0], req_memory_range[1])
  max_latency = np.random.uniform(max_latency_range[0], max_latency_range[1])  # Example latency range
  data_volume = np.random.uniform(data_volume_range[0], data_volume_range[1])  # Example data volume range
  instructions_per_unit_data = np.random.randint(instructions_per_unit_data_range[0], instructions_per_unit_data_range[1])
  tasks.append(Task(req_clock_speed, req_memory, max_latency, data_volume, instructions_per_unit_data))


In [8]:
# prompt: now asisgn each task to a uav randomly and divide the uavs into two sets, (task uav which is alloted tasks and remaining as service uav)

# Assign tasks randomly to UAVs
task_assignments = {}
available_uavs = list(range(num_uavs))
for task_idx in range(num_tasks):
  if available_uavs:
    selected_uav = random.choice(available_uavs)
    task_assignments[task_idx] = selected_uav
    available_uavs.remove(selected_uav)

# Divide UAVs into task UAVs and service UAVs
task_uavs = list(task_assignments.values())
service_uavs = [uav_idx for uav_idx in range(num_uavs) if uav_idx not in task_uavs]

print("Task Assignments:", task_assignments)
print("Task UAVs:", task_uavs)
print("Service UAVs:", service_uavs)


Task Assignments: {0: 3, 1: 8, 2: 9, 3: 12, 4: 0}
Task UAVs: [3, 8, 9, 12, 0]
Service UAVs: [1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 18, 19]


In [9]:
# prompt: now check the eligibility of the service uavs for each task. to check the eligibility, we compare the memory and clock_speed requirements of the task with that of uav and also, we compare the req_data_rate of the task with the data rate between the uav and the task uav to which that task is associated with

# Check eligibility of service UAVs for each task
eligible_uavs = {}
for task_idx, task_uav_idx in task_assignments.items():
  eligible_uavs[task_idx] = []
  for service_uav_idx in service_uavs:
    service_uav = uavs[service_uav_idx]
    task = tasks[task_idx]
    # Check resource requirements and data rate
    if (service_uav.memory >= task.req_memory and
        service_uav.clock_speed >= task.req_clock_speed and
        data_rates[service_uav_idx, task_uav_idx] >= 2*task.req_data_rate and
        service_uav.residual_energy >= service_uav.minimum_energy):
      eligible_uavs[task_idx].append(service_uav_idx)

# Print eligible UAVs for each task
for task_idx, eligible in eligible_uavs.items():
  print(f"Eligible UAVs for Task {task_idx}: {eligible}")


Eligible UAVs for Task 0: [1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 18, 19]
Eligible UAVs for Task 1: [1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 18, 19]
Eligible UAVs for Task 2: [1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 18, 19]
Eligible UAVs for Task 3: [1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 18, 19]
Eligible UAVs for Task 4: [1, 2, 4, 5, 6, 7, 10, 11, 13, 14, 15, 16, 17, 18, 19]


In [10]:
# prompt: print fraction of uavs eligible for each task

# Calculate and print fraction of eligible UAVs for each task
for task_idx, eligible in eligible_uavs.items():
  fraction_eligible = len(eligible) / len(service_uavs) if service_uavs else 0
  print(f"Fraction of eligible UAVs for Task {task_idx}: {fraction_eligible:.2f}")


Fraction of eligible UAVs for Task 0: 1.00
Fraction of eligible UAVs for Task 1: 1.00
Fraction of eligible UAVs for Task 2: 1.00
Fraction of eligible UAVs for Task 3: 1.00
Fraction of eligible UAVs for Task 4: 1.00


In [11]:
# # prompt: save it into a df with two columns, num_tasks and eligible_fraction

# import pandas as pd
# eligible_fractions = []
# for task_idx, eligible in eligible_uavs.items():
#   fraction_eligible = len(eligible) / len(service_uavs) if service_uavs else 0
#   eligible_fractions.append(fraction_eligible)

# df = pd.DataFrame({
#     'num_tasks': range(1, num_tasks + 1),
#     'eligible_fraction': eligible_fractions
# })

# print(df)


In [12]:
# prompt: get the reputation of only service uavs

service_uav_reputations = {uav_idx: uav_reputations[uav_idx] for uav_idx in service_uavs}
print("Service UAV Reputations:")
for uav_idx, reputation in service_uav_reputations.items():
  print(f"UAV {uav_idx}: {reputation}")


Service UAV Reputations:
UAV 1: 195
UAV 2: 52
UAV 4: 132
UAV 5: 92
UAV 6: 168
UAV 7: 119
UAV 10: 168
UAV 11: 162
UAV 13: 165
UAV 14: 100
UAV 15: 28
UAV 16: 208
UAV 17: 33
UAV 18: 144
UAV 19: 10


In [13]:
# prompt: for every task, calculate the trust between the uav associated with task and every other uav. trust = energy od the uav*(r*reputation + d*datarate between them)

# Calculate trust values for each task
trust_values = {}
for task_idx, task_uav_idx in task_assignments.items():
  trust_values[task_idx] = {}
  for service_uav_idx in service_uavs:
    service_uav = uavs[service_uav_idx]
    reputation = service_uav_reputations[service_uav_idx]
    data_rate = data_rates[task_uav_idx, service_uav_idx]
    trust = service_uav.residual_energy * (r * reputation + d * data_rate)
    trust_values[task_idx][service_uav_idx] = trust

# Print trust values for each task
for task_idx, trust_dict in trust_values.items():
  print(f"Trust Values for Task {task_idx}:")
  for service_uav_idx, trust in trust_dict.items():
    print(f"  Service UAV {service_uav_idx}: {trust:.2f}")


Trust Values for Task 0:
  Service UAV 1: 594570558.62
  Service UAV 2: 930403534.05
  Service UAV 4: 701672114.69
  Service UAV 5: 783419463.99
  Service UAV 6: 2373840447.53
  Service UAV 7: 710219106.43
  Service UAV 10: 208775722.26
  Service UAV 11: 103230637.69
  Service UAV 13: 1156120052.11
  Service UAV 14: 265207820.80
  Service UAV 15: 266914430.79
  Service UAV 16: 498363427.23
  Service UAV 17: 272050621.65
  Service UAV 18: 529194470.31
  Service UAV 19: 183955201.70
Trust Values for Task 1:
  Service UAV 1: 631855317.25
  Service UAV 2: 1523959746.20
  Service UAV 4: 914667826.99
  Service UAV 5: 1380419093.74
  Service UAV 6: 2983059651.57
  Service UAV 7: 769578213.64
  Service UAV 10: 271776251.51
  Service UAV 11: 125648694.17
  Service UAV 13: 846418239.75
  Service UAV 14: 329055929.87
  Service UAV 15: 327981931.01
  Service UAV 16: 608060904.79
  Service UAV 17: 389296839.39
  Service UAV 18: 459180414.55
  Service UAV 19: 239257155.88
Trust Values for Task 2:
  

In [14]:
# # prompt: for each task, randomly select a uav from the eligible set for each task (independent of trust).

# # Random UAV selection for each task
# selected_service_uavs = {}
# for task_idx, eligible in eligible_uavs.items():
#   if eligible:
#     selected_service_uavs[task_idx] = random.choice(eligible)

# # Print selected UAVs for each task
# print("Selected UAVs (Random):")
# for task_idx, service_uav_idx in selected_service_uavs.items():
#   print(f"Task {task_idx}: Service UAV {service_uav_idx}")


In [15]:
# prompt: calculate the transmission delay, propagation delay and energy required to transmit a task from task uav to every other service uav

# Calculate transmission, propagation delays, and energy consumption for each task and service UAV
for task_idx, task_uav_idx in task_assignments.items():
  task = tasks[task_idx]
  task_uav = uavs[task_uav_idx]
  print(f"\nTask {task_idx} (Assigned to UAV {task_uav_idx}):")
  for service_uav_idx in service_uavs:
    service_uav = uavs[service_uav_idx]
    distance = calculate_distance(task_uav, service_uav)
    data_rate = data_rates[task_uav_idx, service_uav_idx]

    # Transmission Delay
    transmission_delay = task.data_volume*1024*1024 / data_rate

    # Propagation Delay
    propagation_delay = distance / SPEED_OF_LIGHT

    # Energy Consumption (assuming simple linear model)
    energy_consumption = TRANSMISSION_POWER * transmission_delay

    print(f"  Service UAV {service_uav_idx}:")
    print(f"    Transmission Delay: {transmission_delay} seconds")
    print(f"    Propagation Delay: {propagation_delay} seconds")
    print(f"    Energy Consumption: {energy_consumption} Joules")


Task 0 (Assigned to UAV 3):
  Service UAV 1:
    Transmission Delay: 1.5702994171633162 seconds
    Propagation Delay: 9.813593699542848e-07 seconds
    Energy Consumption: 1.5702994171633162 Joules
  Service UAV 2:
    Transmission Delay: 0.9419721357215336 seconds
    Propagation Delay: 6.554425681746062e-07 seconds
    Energy Consumption: 0.9419721357215336 Joules
  Service UAV 4:
    Transmission Delay: 1.7281284631261986 seconds
    Propagation Delay: 1.0497429047293359e-06 seconds
    Energy Consumption: 1.7281284631261986 Joules
  Service UAV 5:
    Transmission Delay: 0.877882126624939 seconds
    Propagation Delay: 6.152128842952055e-07 seconds
    Energy Consumption: 0.877882126624939 Joules
  Service UAV 6:
    Transmission Delay: 0.829023498737019 seconds
    Propagation Delay: 5.833120879874044e-07 seconds
    Energy Consumption: 0.829023498737019 Joules
  Service UAV 7:
    Transmission Delay: 1.783207217181416 seconds
    Propagation Delay: 1.0726991996211192e-06 second

In [16]:
# # prompt: for each task, select the service uav with maximum trust. note that once the service uav is selected, then do not consider it for the allocating another task

# # Select service UAV with maximum trust for each task (without repetition)
# selected_service_uavs = {}
# used_service_uavs = set()

# for task_idx, trust_dict in trust_values.items():
#   # Sort eligible UAVs by trust in descending order
#   sorted_uavs = sorted(trust_dict, key=trust_dict.get, reverse=True)

#   for service_uav_idx in sorted_uavs:
#     if service_uav_idx not in used_service_uavs:
#       selected_service_uavs[task_idx] = service_uav_idx
#       used_service_uavs.add(service_uav_idx)
#       uavs[service_uav_idx].select_count += 1 # increment select count if uav is selected
#       break

# # Print the selected service UAV for each task
# print("\nSelected Service UAVs:")
# for task_idx, service_uav_idx in selected_service_uavs.items():
#   print(f"Task {task_idx}: Service UAV {service_uav_idx}")


In [ ]:
# prompt: for each task, select the service uav wiith minimum (total delay + energy consumption)

selected_service_uavs = {}
already_selected_uavs = set()

for task_idx, task_uav_idx in task_assignments.items():
    task = tasks[task_idx]
    task_uav = uavs[task_uav_idx]
    min_cost = float('inf')
    selected_uav = None

    for service_uav_idx in service_uavs:
        if service_uav_idx in already_selected_uavs:
            continue  # Skip already selected UAVs

        service_uav = uavs[service_uav_idx]
        distance = calculate_distance(task_uav, service_uav)
        data_rate = data_rates[task_uav_idx, service_uav_idx]

        transmission_delay = task.data_volume * 1024 * 1024 / data_rate
        propagation_delay = distance / SPEED_OF_LIGHT
        energy_consumption = TRANSMISSION_POWER * transmission_delay

        total_cost = transmission_delay + propagation_delay + energy_consumption

        if total_cost < min_cost:
            min_cost = total_cost
            selected_uav = service_uav_idx

    if selected_uav is not None:
        selected_service_uavs[task_idx] = selected_uav
        already_selected_uavs.add(selected_uav)  # Mark UAV as selected

# Print the selected service UAV for each task
print("\nSelected Service UAVs (Minimum Cost):")
for task_idx, service_uav_idx in selected_service_uavs.items():
  print(f"Task {task_idx}: Service UAV {service_uav_idx}")



Selected Service UAVs (Minimum Cost):
Task 0: Service UAV 19
Task 1: Service UAV 14
Task 2: Service UAV 4
Task 3: Service UAV 11
Task 4: Service UAV 16


In [ ]:
print(selected_service_uavs.items())

dict_items([(0, 15), (1, 12), (2, 7), (3, 12), (4, 5)])


In [ ]:
# prompt: calculate discounting factor for each uav where discounting_factor = (total_reputation/reputation_of_uav), then scale it down to 0 to 4 and then apply sigmoid to get the range between 0 to 1

# Calculate total reputation
total_reputation = sum(uav_reputations.values())

# Calculate discounting factors
discounting_factors = {}
for uav_idx, reputation in uav_reputations.items():
  discounting_factors[uav_idx] = total_reputation / reputation

# Scale down to 0 to 4 range
scaled_factors = {}
max_factor = max(discounting_factors.values())
for uav_idx, factor in discounting_factors.items():
  scaled_factors[uav_idx] = (factor / max_factor) * 4

# Apply sigmoid function
sigmoid_factors = {}
for uav_idx, factor in scaled_factors.items():
  sigmoid_factors[uav_idx] = 1 / (1 + np.exp(-factor))

# Print sigmoid discounting factors
print("\nSigmoid Discounting Factors:")
for uav_idx, factor in sigmoid_factors.items():
  print(f"UAV {uav_idx}: {factor:.4f}")



Sigmoid Discounting Factors:
UAV 0: 0.6900
UAV 1: 0.5987
UAV 2: 0.9077
UAV 3: 0.8211
UAV 4: 0.6139
UAV 5: 0.9022
UAV 6: 0.7066
UAV 7: 0.6206
UAV 8: 0.8411
UAV 9: 0.7216
UAV 10: 0.9820
UAV 11: 0.9509
UAV 12: 0.9077
UAV 13: 0.6608
UAV 14: 0.8941
UAV 15: 0.9722
UAV 16: 0.6742
UAV 17: 0.8941
UAV 18: 0.9743
UAV 19: 0.8411


In [ ]:
# prompt: calculate the award to be given by the task uav to service uav for completing a task. award = (1-discounting factor of service uav)/(1-discounting factor of service uav*discounting factor of task uav)

# Calculate awards for selected service UAVs
awards = {}
for task_idx, service_uav_idx in selected_service_uavs.items():
  task_uav_idx = task_assignments[task_idx]
  discounting_factor_service = sigmoid_factors[service_uav_idx]
  discounting_factor_task = sigmoid_factors[task_uav_idx]
  award = (1 - discounting_factor_service) / (1 - discounting_factor_service * discounting_factor_task)
  awards[task_idx] = {
      'service_uav': service_uav_idx,
      'award': award
  }

# Print the awards
print("\nAwards for Service UAVs:")
for task_idx, award_info in awards.items():
  print(f"Task {task_idx}: Service UAV {award_info['service_uav']} receives an award of {award_info['award']:.4f}")



Awards for Service UAVs:
Task 0: Service UAV 15 receives an award of 0.0806
Task 1: Service UAV 12 receives an award of 0.3903
Task 2: Service UAV 7 receives an award of 0.8523
Task 3: Service UAV 12 receives an award of 0.2676
Task 4: Service UAV 5 receives an award of 0.2192


In [ ]:
# prompt: now transfer the score, for transfering the score, the reputation of the task uav which gave the task should reduce by this fraction of award, i.e.  reputation of the task uav =  reputation of the task uav(1-award) and reputation of the service uav =  reputation of the service uav+  reputation of the task uav*award

# Transfer scores based on awards
for task_idx, award_info in awards.items():
  service_uav_idx = award_info['service_uav']
  task_uav_idx = task_assignments[task_idx]
  award = award_info['award']

  # Update reputations
  temp = uav_reputations[task_uav_idx]
  uav_reputations[task_uav_idx] *= (1 - award)
  uav_reputations[service_uav_idx] += uav_reputations[task_uav_idx] * award

# Print updated reputations
print("\nUpdated UAV Reputations:")
for uav_idx, reputation in uav_reputations.items():
  print(f"UAV {uav_idx}: {reputation:.2f}")



Updated UAV Reputations:
UAV 0: 200.00
UAV 1: 400.00
UAV 2: 70.00
UAV 3: 105.00
UAV 4: 269.39
UAV 5: 131.04
UAV 6: 182.00
UAV 7: 334.44
UAV 8: 96.00
UAV 9: 123.05
UAV 10: 40.00
UAV 11: 54.00
UAV 12: 125.77
UAV 13: 240.00
UAV 14: 75.00
UAV 15: 61.30
UAV 16: 202.27
UAV 17: 11.08
UAV 18: 44.00
UAV 19: 58.53


In [ ]:
# Update UAV positions after an iteration of the task
update_uav_positions(uavs, LONG, LAT, ALT)

# Print updated positions for verification
for i, uav in enumerate(uavs):
    print(f"UAV {i+1}: Longitude = {uav.longitude}, Latitude = {uav.latitude}, Altitude = {uav.altitude}")

UAV 1: Longitude = 61.18573638301324, Latitude = 65.70700341881385, Altitude = 6.1345593345552905
UAV 2: Longitude = 99.79685336443845, Latitude = 42.926597146636944, Altitude = 11.040201812195084
UAV 3: Longitude = 1.1046779633702375, Latitude = 26.65089379890948, Altitude = 16.29249624268516
UAV 4: Longitude = 10.376838298080449, Latitude = 87.55392021140622, Altitude = 16.748949593632574
UAV 5: Longitude = 42.33910487198557, Latitude = 73.4297739766412, Altitude = 19.83335622267191
UAV 6: Longitude = 44.9098050478572, Latitude = 32.08067069831198, Altitude = 12.801981201949134
UAV 7: Longitude = 74.4185361871268, Latitude = 18.333484692190382, Altitude = 9.123219340681903
UAV 8: Longitude = 40.66298842497857, Latitude = 64.14693686237162, Altitude = 12.294566771982554
UAV 9: Longitude = 46.14584342141508, Latitude = 68.77200677457776, Altitude = 18.777883355692225
UAV 10: Longitude = 46.178609491977355, Latitude = 28.972458338999264, Altitude = 13.967730555113853
UAV 11: Longitude =

In [ ]:
# prompt: recalculate the residual energy of the service uav by reducing it by avg power * time of execution. and also reduct the energy of task uav the enrgy required to transmit the task, you have the task size and all the other parameters, it is upto you if any other variable that are needed to be declared.

# Calculate execution times for tasks
total_energy_consumed = 0
execution_times = {}
transmission_times = {}
for task_idx, service_uav_idx in selected_service_uavs.items():
  task = tasks[task_idx]
  service_uav = uavs[service_uav_idx]
  execution_time = task.total_instructions / (service_uav.clock_speed*1e6)
  execution_time = execution_time / 3600
  execution_times[task_idx] = execution_time

  # Reduce service UAV's residual energy
  energy_consumed_service = service_uav.avg_power_consumption * execution_time
  total_energy_consumed += energy_consumed_service
  uavs[service_uav_idx].residual_energy -= energy_consumed_service

  # Calculate energy consumed by task UAV for transmission (simplified model)
  task_uav_idx = task_assignments[task_idx]
  distance = calculate_distance(uavs[task_uav_idx], uavs[service_uav_idx])
  transmission_time = (task.data_volume*1024*1024) / calculate_data_rate(distance)
  transmission_times[task_idx] = transmission_time
  energy_consumed_task = TRANSMISSION_POWER * transmission_time
  total_energy_consumed += energy_consumed_task
  uavs[task_uav_idx].residual_energy -= energy_consumed_task

# Print execution times
print("\nTask Execution Times:")
for task_idx, time in execution_times.items():
  print(f"Task {task_idx}: {time:.4f} hours")

# Print updated residual energies
print("\nUpdated Residual Energies:")
for i, uav in enumerate(uavs):
  print(f"UAV {i}: {uav.residual_energy:.2f} Watt-hrs")

print("Total energy consumed: ", total_energy_consumed)


Task Execution Times:
Task 0: 0.0136 hours
Task 1: 0.0115 hours
Task 2: 0.0051 hours
Task 3: 0.0844 hours
Task 4: 0.0237 hours

Updated Residual Energies:
UAV 0: 55.39 Watt-hrs
UAV 1: 80.17 Watt-hrs
UAV 2: 71.92 Watt-hrs
UAV 3: 93.79 Watt-hrs
UAV 4: 132.80 Watt-hrs
UAV 5: 51.25 Watt-hrs
UAV 6: 126.00 Watt-hrs
UAV 7: 165.95 Watt-hrs
UAV 8: 52.88 Watt-hrs
UAV 9: 155.14 Watt-hrs
UAV 10: 66.13 Watt-hrs
UAV 11: 104.51 Watt-hrs
UAV 12: 131.98 Watt-hrs
UAV 13: 191.41 Watt-hrs
UAV 14: 153.87 Watt-hrs
UAV 15: 77.69 Watt-hrs
UAV 16: 156.46 Watt-hrs
UAV 17: 69.85 Watt-hrs
UAV 18: 89.99 Watt-hrs
UAV 19: 145.17 Watt-hrs
Total energy consumed:  38.03930760097714


In [ ]:
# prompt: create a dataframe named uav for all the uavs and the components of uav class as the columns

import pandas as pd
# Create a list of dictionaries, where each dictionary represents a UAV
uav_data = []
for i, uav in enumerate(uavs):
  uav_data.append({
      'UAV ID': i,
      'Residual Energy': uav.residual_energy,
      'Memory': uav.memory,
      'Clock Speed': uav.clock_speed,
      'Longitude': uav.longitude,
      'Latitude': uav.latitude,
      'Altitude': uav.altitude,
      'Avg Power Consumption': uav.avg_power_consumption,
      'Minimum Energy': uav.minimum_energy
    #   'Transmission Times': transmission_times.items()[1],
    #   'Execution Times': execution_times.items()[1]
  })

# Create the DataFrame from the list of dictionaries
uav_df = pd.DataFrame(uav_data)

# Display the DataFrame
print(uav_df)

TypeError: 'dict_items' object is not subscriptable